In [1]:
import pandas as pd
import numpy as np

# Load data
df = pd.read_csv("../data/nycparking2025.csv")

# Clean column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()


Shape: (7057515, 43)
Columns: ['summons_number', 'plate_id', 'registration_state', 'plate_type', 'issue_date', 'violation_code', 'vehicle_body_type', 'vehicle_make', 'issuing_agency', 'street_code1', 'street_code2', 'street_code3', 'vehicle_expiration_date', 'violation_location', 'violation_precinct', 'issuer_precinct', 'issuer_code', 'issuer_command', 'issuer_squad', 'violation_time', 'time_first_observed', 'violation_county', 'violation_in_front_of_or_opposite', 'house_number', 'street_name', 'intersecting_street', 'date_first_observed', 'law_section', 'sub_division', 'violation_legal_code', 'days_parking_in_effect', 'from_hours_in_effect', 'to_hours_in_effect', 'vehicle_color', 'unregistered_vehicle?', 'vehicle_year', 'meter_number', 'feet_from_curb', 'violation_post_code', 'violation_description', 'no_standing_or_stopping_violation', 'hydrant_violation', 'double_parking_violation']


,summons_number,plate_id,registration_state,plate_type,issue_date,violation_code,vehicle_body_type,vehicle_make,issuing_agency,street_code1,...,vehicle_color,unregistered_vehicle?,vehicle_year,meter_number,feet_from_curb,violation_post_code,violation_description,no_standing_or_stopping_violation,hydrant_violation,double_parking_violation
0,9139716661,LKS7820,NY,PAS,07/02/2024,38,SUBN,TOYOT,T,26790,...,WH,NaN,2024.0,100038,0.0,71.0,38-Failure to Dsplay Meter Rec,NaN,NaN,NaN
1,4903865265,Y23UMS,NJ,PAS,06/21/2024,36,VAN,ME/BE,V,0,...,NaN,NaN,2024.0,NaN,0.0,NaN,PHTO SCHOOL ZN SPEED VIOLATION,NaN,NaN,NaN
2,9140510268,N76NJN,NJ,PAS,07/08/2024,14,SUBN,HYUND,T,24890,...,WHITE,NaN,0.0,NaN,0.0,12.0,14-No Standing,NaN,NaN,NaN
3,9136561575,42UD91,NY,MCL,07/12/2024,71,MCY,HONDA,T,47030,...,BLACK,NaN,0.0,NaN,0.0,9.0,71A-Insp Sticker Expired (NYS),NaN,NaN,NaN
4,9139621832,76925NE,NY,COM,07/11/2024,14,VAN,ISUZU,T,51090,...,WH,NaN,2018.0,NaN,0.0,12.0,14-No Standing,NaN,NaN,NaN


In [2]:
# Inspect nulls and zero values

def inspect_columns(df, columns=None):
    if columns is None:
        columns = df.columns

    report = []
    for col in columns:
        s = df[col]
        null_count = s.isna().sum()

        zero_count = None
        if pd.api.types.is_numeric_dtype(s):
            zero_count = (s == 0).sum()

        report.append({
            "column": col,
            "dtype": str(s.dtype),
            "rows": len(s),
            "null_count": int(null_count),
            "null_pct": round(null_count / len(s) * 100, 2),
            "zero_count": int(zero_count) if zero_count is not None else None,
            "non_null_count": int(s.notna().sum())
        })

    return pd.DataFrame(report).sort_values(["null_count", "column"], ascending=[False, True])

report = inspect_columns(df)
print(report.to_string(index=False))


                           column   dtype    rows  null_count  null_pct  zero_count  non_null_count
         double_parking_violation float64 7057515     7057515    100.00         0.0               0
                hydrant_violation float64 7057515     7057515    100.00         0.0               0
no_standing_or_stopping_violation float64 7057515     7057515    100.00         0.0               0
            unregistered_vehicle? float64 7057515     6930572     98.20    126943.0          126943
              time_first_observed     str 7057515     6629386     93.93         NaN          428129
                     meter_number     str 7057515     6180579     87.57         NaN          876936
               to_hours_in_effect     str 7057515     4852748     68.76         NaN         2204767
             from_hours_in_effect     str 7057515     4852738     68.76         NaN         2204777
              violation_post_code float64 7057515     4589360     65.03         0.0         2468155


In [3]:
# Find columns with many nulls
high_null_cols = report[report["null_pct"] > 50]
print("Columns with >50% nulls:")
print(high_null_cols.to_string(index=False))

# Find numeric columns with many zeros
zero_report = report[report["zero_count"].notna()].copy()
zero_report["zero_pct"] = (zero_report["zero_count"] / zero_report["rows"] * 100).round(2)

high_zero_cols = zero_report[zero_report["zero_pct"] > 50]
print("\nNumeric columns with >50% zeros:")
print(high_zero_cols[["column", "dtype", "zero_count", "zero_pct"]].to_string(index=False))


Columns with >50% nulls:
                           column   dtype    rows  null_count  null_pct  zero_count  non_null_count
         double_parking_violation float64 7057515     7057515    100.00         0.0               0
                hydrant_violation float64 7057515     7057515    100.00         0.0               0
no_standing_or_stopping_violation float64 7057515     7057515    100.00         0.0               0
            unregistered_vehicle? float64 7057515     6930572     98.20    126943.0          126943
              time_first_observed     str 7057515     6629386     93.93         NaN          428129
                     meter_number     str 7057515     6180579     87.57         NaN          876936
               to_hours_in_effect     str 7057515     4852748     68.76         NaN         2204767
             from_hours_in_effect     str 7057515     4852738     68.76         NaN         2204777
              violation_post_code float64 7057515     4589360     65.03    

## Feature Selection and Data Reduction

The original NYC parking violations dataset contained 43 columns and over 7 million records. A profiling analysis was conducted to evaluate null percentages, sparsity, and analytical relevance.

Columns with extremely high missing-value percentages, operational metadata fields, and low-value enforcement attributes were removed prior to database import. This reduced dataset complexity, improved query performance, and optimized storage efficiency while preserving columns relevant to enforcement, geographic, and demographic analysis.

In [4]:
import pandas as pd

df = pd.read_csv("../data/nycparking2025.csv", nrows=5)

print(df.columns.tolist())

['Summons Number', 'Plate ID', 'Registration State', 'Plate Type', 'Issue Date', 'Violation Code', 'Vehicle Body Type', 'Vehicle Make', 'Issuing Agency', 'Street Code1', 'Street Code2', 'Street Code3', 'Vehicle Expiration Date', 'Violation Location', 'Violation Precinct', 'Issuer Precinct', 'Issuer Code', 'Issuer Command', 'Issuer Squad', 'Violation Time', 'Time First Observed', 'Violation County', 'Violation In Front Of Or Opposite', 'House Number', 'Street Name', 'Intersecting Street', 'Date First Observed', 'Law Section', 'Sub Division', 'Violation Legal Code', 'Days Parking In Effect', 'From Hours In Effect', 'To Hours In Effect', 'Vehicle Color', 'Unregistered Vehicle?', 'Vehicle Year', 'Meter Number', 'Feet From Curb', 'Violation Post Code', 'Violation Description', 'No Standing or Stopping Violation', 'Hydrant Violation', 'Double Parking Violation']


In [ ]:
from pathlib import Path
import pandas as pd
import csv

INPUT_CSV = Path(r"C:\Users\jayson.coker\Documents\nycparking\data\nycparking2025.csv")
OUTPUT_CSV = Path(r"C:\Users\jayson.coker\Documents\nycparking\data\parking_clean.csv")

CHUNK_SIZE = 100_000

# =========================
# ORIGINAL COLUMNS
# =========================

COLUMNS_TO_KEEP = [
    'Summons Number',
    'Plate ID',
    'Registration State',
    'Plate Type',
    'Issue Date',
    'Violation Code',
    'Vehicle Body Type',
    'Vehicle Make',
    'Issuing Agency',
    'Violation Precinct',
    'Issuer Precinct',
    'Violation Time',
    'Violation County',
    'Street Name',
    'Vehicle Color',
    'Vehicle Year',
    'Violation Description'
]

print("Starting export...")

first_chunk = True
total_rows = 0

for i, chunk in enumerate(
    pd.read_csv(
        INPUT_CSV,
        usecols=COLUMNS_TO_KEEP,
        chunksize=CHUNK_SIZE,
        low_memory=False
    )
):

    print(f"Processing chunk {i + 1}")

    # =========================
    # CLEAN COLUMN NAMES
    # =========================
    chunk.columns = (
        chunk.columns
        .str.lower()
        .str.strip()
        .str.replace(" ", "_")
        .str.replace("?", "", regex=False)
    )

    # =========================
    # CLEAN DATES
    # =========================
    chunk["issue_date"] = pd.to_datetime(
        chunk["issue_date"],
        errors="coerce"
    )

    chunk = chunk.dropna(subset=["issue_date"])
    chunk["issue_date"] = chunk["issue_date"].dt.strftime("%Y-%m-%d")

    # =========================
    # CLEAN TEXT FIELDS
    # =========================
    text_cols = [
        "plate_id",
        "registration_state",
        "plate_type",
        "vehicle_body_type",
        "vehicle_make",
        "issuing_agency",
        "violation_time",
        "violation_county",
        "street_name",
        "vehicle_color",
        "violation_description"
    ]

    for col in text_cols:
        chunk[col] = (
            chunk[col]
            .astype(str)
            .str.strip()
            .str.upper()
        )

    # =========================
    # SAFE CSV EXPORT (IMPORTANT FIX)
    # =========================
    chunk.to_csv(
        OUTPUT_CSV,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False,
        quoting=csv.QUOTE_MINIMAL,
        encoding="utf-8"
    )

    total_rows += len(chunk)

    print(f"Saved chunk {i + 1} | Total rows: {total_rows:,}")

    first_chunk = False

print("\nDONE")
print(f"Output: {OUTPUT_CSV}")
print(f"Total rows exported: {total_rows:,}")

Starting export...
Processing chunk 1
Saved chunk 1 | Total rows: 99,993
Processing chunk 2
Saved chunk 2 | Total rows: 199,984
Processing chunk 3
Saved chunk 3 | Total rows: 299,975
Processing chunk 4
Saved chunk 4 | Total rows: 399,962
Processing chunk 5
Saved chunk 5 | Total rows: 499,958
Processing chunk 6
Saved chunk 6 | Total rows: 599,949
Processing chunk 7
Saved chunk 7 | Total rows: 699,941
Processing chunk 8
Saved chunk 8 | Total rows: 799,934
Processing chunk 9
Saved chunk 9 | Total rows: 899,925
Processing chunk 10
Saved chunk 10 | Total rows: 999,912
Processing chunk 11
Saved chunk 11 | Total rows: 1,099,897
Processing chunk 12
Saved chunk 12 | Total rows: 1,199,883
Processing chunk 13
Saved chunk 13 | Total rows: 1,299,871
Processing chunk 14
Saved chunk 14 | Total rows: 1,399,861
Processing chunk 15
Saved chunk 15 | Total rows: 1,499,855
Processing chunk 16
Saved chunk 16 | Total rows: 1,599,847
Processing chunk 17
Saved chunk 17 | Total rows: 1,699,838
Processing chunk 

In [1]:
with open(r"C:\csv\parking_clean.csv", "r", encoding="utf-8") as f:
    for i in range(20):
        print(repr(f.readline()))

'"summons_number","plate_id","registration_state","plate_type","issue_date","violation_code","vehicle_body_type","vehicle_make","issuing_agency","violation_precinct","issuer_precinct","violation_time","violation_county","street_name","vehicle_color","vehicle_year","violation_description"\n'
'"9139716661","LKS7820","NY","PAS","2024-07-02","38","SUBN","TOYOT","T","5","5","0209P","NY","MOTT ST","WH","2024","38-FAILURE TO DSPLAY METER REC"\n'
'"4903865265","Y23UMS","NJ","PAS","2024-06-21","36","VAN","ME/BE","V","0","0","1009A","BK","NB UTICA AVE @ AVE L","","2024","PHTO SCHOOL ZN SPEED VIOLATION"\n'
'"9140510268","N76NJN","NJ","PAS","2024-07-08","14","SUBN","HYUND","T","18","18","0325P","NY","LEXINGTON AVE","WHITE","0","14-NO STANDING"\n'
'"9136561575","42UD91","NY","MCL","2024-07-12","71","MCY","HONDA","T","79","79","0939A","K","HANCOCK ST","BLACK","0","71A-INSP STICKER EXPIRED (NYS)"\n'
'"9139621832","76925NE","NY","COM","2024-07-11","14","VAN","ISUZU","T","103","103","1116A","Q","JAMAIC

In [2]:
import pandas as pd

df = pd.read_csv(r"C:\csv\parking_clean.csv", nrows=5)
print(df[['summons_number', 'violation_description']])

   summons_number           violation_description
0      9139716661  38-FAILURE TO DSPLAY METER REC
1      4903865265  PHTO SCHOOL ZN SPEED VIOLATION
2      9140510268                  14-NO STANDING
3      9136561575  71A-INSP STICKER EXPIRED (NYS)
4      9139621832                  14-NO STANDING
